# Raw Data Analysis

**Purpose** : this notebook loads individual behavioural data into a common dataset, preprocesses it, uses pre-registered inclusion/exclusion criteria, and finally applies basic checks to ensure data integrity. The sanity checks applied are:
1. Outcome probabilities observed, relative to design
2. Correct OC/R property swap between Task 1 and Task 2
3. Colour values swapping between Task 1 and Task 2, whilst the SC property should stay constant
4. Randomisation balance and Block 2 exposure counts appropriate to design

**Source** : Prolific dataset (n = 80), collected Thursday 2nd April, 2026.     
**OSF project link** : https://osf.io/fkb87     
**OSF pre-registration link** : https://osf.io/uj92x/overview

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

## Load & Clean

In [3]:
# Load and concatenate raw data

data_dir = PROJECT_ROOT / "data" / "raw"

dfs = []
excluded = []

for f in data_dir.glob("*.csv"):
    subject_df = pd.read_csv(f)
    is_test = f.name.startswith("test_")
    wrong_length = len(subject_df) != 728

    if is_test or wrong_length:
        excluded.append((f.name, len(subject_df)))
        continue

    debrief = subject_df[subject_df['phase'] == 'debrief']
    completion_ms = debrief['time_elapsed'].values[0] if len(debrief) > 0 else np.nan
    subject_df['completion_ms'] = completion_ms
    dfs.append(subject_df)

raw = pd.concat(dfs, ignore_index=True)
outcome_df = raw[raw['phase'] == 'outcome'].copy()

print(f'Included files: {len(dfs)}')
print(f'Excluded files: {len(excluded)}')
print()
print(f'Total raw rows: {len(raw)}')
print(f'Included rows: {len(outcome_df)}')
print()
print(f'Subjects (before exclusions): {raw["subject_id"].nunique()}')

# Drop uninformative + Prolific-native rows
drop_cols = ['stimulus', 'response', 'trial_type', 'trial_index', 'time_elapsed', 'internal_node_id']
outcome_df = outcome_df.drop(columns=drop_cols)

# Data types
outcome_df['pressed'] = pd.to_numeric(outcome_df['pressed'], errors='coerce').astype('Int64')
outcome_df['outcome'] = pd.to_numeric(outcome_df['outcome'], errors='coerce').astype('Int64')
outcome_df['outcome_shown'] = pd.to_numeric(outcome_df['outcome_shown'], errors='coerce').astype('Int64')
outcome_df['watch_only'] = pd.to_numeric(outcome_df['watch_only'], errors='coerce').astype('Int64')
outcome_df['rt'] = pd.to_numeric(outcome_df['rt'], errors='coerce')  # null should return NaN, not 0
outcome_df['task'] = pd.to_numeric(outcome_df['task'], errors='coerce').astype('Int64')
outcome_df['block'] = pd.to_numeric(outcome_df['block'], errors='coerce').astype('Int64')
outcome_df['trial_n'] = pd.to_numeric(outcome_df['trial_n'], errors='coerce').astype('Int64')


Included files: 74
Excluded files: 23

Total raw rows: 53872
Included rows: 13320

Subjects (before exclusions): 74


## Pre-Registered Exclusions

**OSF pre-registration** : pre-registered exclusion criteria listed below.
- Completion time under 10 minutes (insufficient engagement)
- Overall press rate below 5% or above 95% across all trials (non-engagement or perseveration)
- More than 20% of trials with missing press data
- Outlier subjects: press rates more than 3 SD from the group mean on any primary measure will be flagged and reported but not automatically excluded — sensitivity analyses will be run with and without these subjects

In [4]:
# Completion time < 10 minutes
outcome_df['completion_min'] = outcome_df['completion_ms'] / 60000
exclude_time = outcome_df.groupby('subject_id')['completion_min'].first()
exclude_time = exclude_time[exclude_time < 10].index
print(f'Excluded (completion < 10min): {len(exclude_time)}')

# Press rate < 5% or > 95%
press_rate = outcome_df[outcome_df['watch_only'] == 0].groupby('subject_id')['pressed'].mean()
exclude_press = press_rate[(press_rate < 0.05) | (press_rate > 0.95)].index
print(f'Excluded (press rate floor/ceiling): {len(exclude_press)}')

# Missing trial data (> 20% outcome rows missing pressed value)
missing_rate = outcome_df.groupby('subject_id')['pressed'].apply(lambda x: x.isna().mean())
exclude_missing = missing_rate[missing_rate > 0.2].index
print(f'Excluded (missing data): {len(exclude_missing)}')

# Flag, but do not exclude, subjects > 3 SD from group mean overall press rate
press_rate_mean = press_rate.mean()
press_rate_sd = press_rate.std()
outlier_subjects = press_rate[
    (press_rate - press_rate_mean).abs() > 3 * press_rate_sd
].index
print(f'Flagged (press rate outliers): {len(outlier_subjects)}')
print()
# Combine exclusions
all_excluded = set(exclude_time) | set(exclude_press) | set(exclude_missing)
print(f'Total excluded subjects: {len(all_excluded)}')
print(f'Remaining subjects: {outcome_df["subject_id"].nunique() - len(all_excluded)}')
print()

if len(all_excluded):
    print('Excluded subject IDs:')
    print(sorted(all_excluded))

df_clean = outcome_df[~outcome_df['subject_id'].isin(all_excluded)].copy()


Excluded (completion < 10min): 0
Excluded (press rate floor/ceiling): 4
Excluded (missing data): 0
Flagged (press rate outliers): 0

Total excluded subjects: 4
Remaining subjects: 70

Excluded subject IDs:
['5a46d551b77a5000014a8ce9', '5ba0a49df074140001058d8d', '612d65fb9c0182abbcf41c32', '698a2abb6a4175298b207fce']


In [5]:
# Reconstruct Task 2 OC/R property labels before saving.

task1_roles = df_clean[df_clean['task'] == 1].groupby('subject_id')[
    ['prop_OC', 'prop_R']
].first().rename(columns={'prop_OC': 'prop_OC_t1', 'prop_R': 'prop_R_t1'})

df_clean = df_clean.merge(task1_roles, on='subject_id', how='left')
df_clean['prop_OC'] = np.where(df_clean['task'] == 2, df_clean['prop_R_t1'], df_clean['prop_OC'])
df_clean['prop_R'] = np.where(df_clean['task'] == 2, df_clean['prop_OC_t1'], df_clean['prop_R'])
df_clean = df_clean.drop(columns=['prop_OC_t1', 'prop_R_t1'])
df_clean = df_clean.sort_values(['subject_id', 'task', 'block', 'trial_n'])
df_clean.to_csv(PROJECT_ROOT / 'data' / 'processed' / 'clean_data.csv', index=False)

print(f'\nClean data saved: {len(df_clean)} rows, {df_clean["subject_id"].nunique()} subjects')
print(f'Columns retained: {list(df_clean.columns)}')


Clean data saved: 12600 rows, 70 subjects
Columns retained: ['rt', 'subject_id', 'group', 'phase', 'task', 'block', 'trial_n', 'watch_only', 'prop_SC', 'prop_OC', 'prop_R', 'trial_colour', 'trial_size', 'trial_velocity', 'trial_role', 'pressed', 'outcome', 'outcome_shown', 'completion_ms', 'completion_min']


In [6]:
trials = df_clean[df_clean['watch_only'] == 0].copy()
trials['pressed'] = pd.to_numeric(trials['pressed'], errors='coerce')
trials['outcome'] = pd.to_numeric(trials['outcome'], errors='coerce')

# 1. Outcome probabilities should match the task design
outcome_rates = trials.groupby(['trial_role', 'pressed'])['outcome'].agg(['mean', 'count']).round(3)
print('Outcome rates by role and action:')
print(outcome_rates)
print()
print("Expected outcome rates (approx):")
print("SC (1) = 0.95, (0) = 0.05")
print("OC (any) = 0.90")
print("R (any) = 0.10")
print()
print()

# 2. Task 2 should swap OC and R properties relative to Task 1
swap_check = df_clean.groupby(['subject_id', 'task'])[['prop_OC', 'prop_R']].first().unstack('task')
swap_check.columns = ['OC_task1', 'OC_task2', 'R_task1', 'R_task2']
swap_check['swap_correct'] = (
    (swap_check['OC_task1'] == swap_check['R_task2']) &
    (swap_check['R_task1'] == swap_check['OC_task2'])
)

# 3. Colour values should stay task-specific; SC property should stay constant
colour_by_task = df_clean.groupby(['subject_id', 'task'])['trial_colour'].unique().reset_index()
task1_colour_ok = colour_by_task.loc[colour_by_task['task'] == 1, 'trial_colour'].apply(
    lambda x: set(x).issubset({'blue', 'red'})
).all()
task2_colour_ok = colour_by_task.loc[colour_by_task['task'] == 2, 'trial_colour'].apply(
    lambda x: set(x).issubset({'green', 'purple'})
).all()

sc_by_task = df_clean.groupby(['subject_id', 'task'])['prop_SC'].first().unstack('task')
sc_by_task.columns = ['SC_task1', 'SC_task2']
sc_constant = (sc_by_task['SC_task1'] == sc_by_task['SC_task2']).all()

print("Sanity checks:")
print(f'OC/R swap correct: {swap_check["swap_correct"].sum()}/{len(swap_check)} subjects')
print(f'Task 1 colours red/blue only: {task1_colour_ok}')
print(f'Task 2 colours green/purple only: {task2_colour_ok}')
print(f'SC property constant across tasks: {sc_constant}')
print()
print()

failed_checks = []
if not swap_check['swap_correct'].all():
    failed_checks.append('OC/R swap')
if not task1_colour_ok:
    failed_checks.append('Task 1 colours')
if not task2_colour_ok:
    failed_checks.append('Task 2 colours')
if not sc_constant:
    failed_checks.append('SC consistency')
if failed_checks:
    raise ValueError(f'Failed sanity checks: {failed_checks}')


# 4. Randomisation balance and Block 2 exposure counts - descriptive diagnostics of trial assignments
subject_props = df_clean.groupby('subject_id').agg(
    group=('group', 'first'),
    sc_prop=('prop_SC', 'first')
)

print('Group x SC property balance:')
print(pd.crosstab(subject_props['group'], subject_props['sc_prop']))
print()
print()

MIN_BLOCK2_ROLE_TRIALS = 4
block2_counts = (
    trials[trials['block'] == 2]
    .groupby(['subject_id', 'group', 'task', 'trial_role'])
    .agg(
        n_trials=('pressed', 'count'),
        n_presses=('pressed', 'sum'),
        press_rate=('pressed', 'mean')
    )
    .reset_index()
)
block2_low_counts = block2_counts[
    block2_counts['n_trials'] < MIN_BLOCK2_ROLE_TRIALS
].sort_values(['n_trials', 'subject_id', 'task', 'trial_role'])

print('Block 2 trial counts per subject-task-role:')
print(block2_counts.groupby('trial_role')['n_trials'].describe()[['min', '25%', '50%', '75%', 'max']].round(1))
print()
print()

print(f'Block 2 cells with fewer than {MIN_BLOCK2_ROLE_TRIALS} trials: {len(block2_low_counts)}')
if len(block2_low_counts):
    print(block2_low_counts.to_string(index=False))

Outcome rates by role and action:
                     mean  count
trial_role pressed              
OC         0        0.908    599
           1        0.899    991
R          0        0.088   1101
           1        0.089    662
SC         0        0.051   4079
           1        0.952   3768

Expected outcome rates (approx):
SC (1) = 0.95, (0) = 0.05
OC (any) = 0.90
R (any) = 0.10


Sanity checks:
OC/R swap correct: 70/70 subjects
Task 1 colours red/blue only: True
Task 2 colours green/purple only: True
SC property constant across tasks: True


Group x SC property balance:
sc_prop  colour  size  velocity
group                          
A            12     9        18
B            10    16         5


Block 2 trial counts per subject-task-role:
             min   25%   50%   75%   max
trial_role                              
OC           2.0   7.0   8.5  11.0  16.0
R            3.0   8.0   9.0  11.0  17.0
SC          33.0  40.0  42.0  45.0  51.0


Block 2 cells with fewer than 4 tr